In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder

np.random.seed(42)

DATA_DIR = "/kaggle/input/osic-pulmonary-fibrosis-progression"

print("Listing a few files under /kaggle/input ...")
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))
    break



Listing a few files under /kaggle/input ...
/kaggle/input/train.csv
/kaggle/input/sample_submission.csv
/kaggle/input/sample_submission.csv.zip
/kaggle/input/test.csv
/kaggle/input/test.csv.zip


In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
train_raw = train.copy()
test = pd.read_csv(f"{DATA_DIR}/test.csv")
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

train.drop(train[train.Patient == "ID00197637202246865691526"].index, inplace=True)

all_patients = pd.concat(
    [train[["Patient"]], test[["Patient"]]], axis=0, ignore_index=True
)
le_id = LabelEncoder()
le_id.fit(all_patients["Patient"])

train["PatientID"] = le_id.transform(train["Patient"])
test["PatientID"] = le_id.transform(test["Patient"])

train.head()




,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,PatientID
0,ID00133637202223847701934,-2,3195,92.856312,83,Male,Never smoked,61
1,ID00133637202223847701934,2,3203,93.088817,83,Male,Never smoked,61
2,ID00133637202223847701934,4,3097,90.008138,83,Male,Never smoked,61
3,ID00133637202223847701934,6,3171,92.158800,83,Male,Never smoked,61
4,ID00133637202223847701934,8,3115,90.531272,83,Male,Never smoked,61


In [3]:
def add_baselines(data: pd.DataFrame) -> pd.DataFrame:
    aux = data[["Patient", "Weeks"]].groupby("Patient").min().reset_index()
    aux = pd.merge(
        aux, data[["Patient", "Weeks", "FVC"]], how="left", on=["Patient", "Weeks"]
    )
    aux = aux.groupby("Patient").mean(numeric_only=True).reset_index()
    aux["Weeks"] = aux["Weeks"].astype(int)
    aux["FVC"] = aux["FVC"].astype(int)
    data = pd.merge(data, aux, how="left", on="Patient", suffixes=("", "_base"))
    return data


train = add_baselines(train)
test = add_baselines(test)

train.head()




,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,PatientID,Weeks_base,FVC_base
0,ID00133637202223847701934,-2,3195,92.856312,83,Male,Never smoked,61,-2,3195
1,ID00133637202223847701934,2,3203,93.088817,83,Male,Never smoked,61,-2,3195
2,ID00133637202223847701934,4,3097,90.008138,83,Male,Never smoked,61,-2,3195
3,ID00133637202223847701934,6,3171,92.158800,83,Male,Never smoked,61,-2,3195
4,ID00133637202223847701934,8,3115,90.531272,83,Male,Never smoked,61,-2,3195


In [4]:
def patient_class(row):
    if row["Sex"] == "Male":
        if row["SmokingStatus"] == "Currently smokes":
            return 0
        elif row["SmokingStatus"] == "Ex-smoker":
            return 1
        elif row["SmokingStatus"] == "Never smoked":
            return 2
    else:
        if row["SmokingStatus"] == "Currently smokes":
            return 3
        elif row["SmokingStatus"] == "Ex-smoker":
            return 4
        elif row["SmokingStatus"] == "Never smoked":
            return 5


train["Class"] = train.apply(patient_class, axis=1)
test["Class"] = test.apply(patient_class, axis=1)

test.head()



,Patient,Weeks,FVC,Percent,Age,Sex,SmokingStatus,PatientID,Weeks_base,FVC_base,Class
0,ID00014637202177757139317,0,3807,90.076661,56,Male,Ex-smoker,5,0,3807,1
1,ID00019637202178323708467,13,2100,92.858722,83,Female,Ex-smoker,7,13,2100,4
2,ID00047637202184938901501,2,3313,89.929425,68,Male,Ex-smoker,18,2,3313,1
3,ID00082637202201836229724,19,2918,99.794802,49,Female,Currently smokes,33,19,2918,3
4,ID00126637202218610655908,18,2375,59.375000,78,Male,Ex-smoker,54,18,2375,1


In [5]:
PatientID = train["Patient"].values
fvc_b = train.groupby("Patient").first()["FVC_base"]
fvc_b.values[:10]




array([2315, 3660, 3523, 3326, 3418, 3042, 2297, 1536, 2903, 2884])

In [6]:
def model_fit(data, examine=True):
    """
    Returns a lightweight 'model' dict holding per-patient (a, b) parameters and
    per-class residual sigma estimates, matching the original hierarchical intent.
    """
    params = {}
    residuals = []
    for pid, g in data.groupby("PatientID"):
        x = g["Weeks"].values.astype(float)
        y = g["FVC"].values.astype(float)

        if len(g) >= 2 and np.std(x) > 0:
            b, a = np.polyfit(x, y, 1)
        else:
            a = float(g["FVC_base"].iloc[0])
            b = 0.0

        params[int(pid)] = (float(a), float(b))

        yhat = a + b * x
        res = y - yhat
        for r, cls in zip(res, g["Class"].values.astype(int)):
            residuals.append((int(cls), float(r)))

    residuals = np.array(residuals, dtype=float) if len(residuals) else np.zeros((0, 2))
    if len(residuals) > 0:
        global_sigma = float(np.std(residuals[:, 1]) + 1e-6)
    else:
        global_sigma = 200.0

    sigma_by_class = {c: global_sigma for c in range(6)}
    if len(residuals) > 10:
        for c in range(6):
            rc = residuals[residuals[:, 0] == c, 1]
            if len(rc) >= 5:
                sigma_by_class[c] = float(np.std(rc) + 1e-6)

    model = {
        "params": params,
        "sigma_by_class": sigma_by_class,
        "global_sigma": global_sigma,
    }
    trace = None  # kept for API compatibility with downstream calls
    return model, trace




In [7]:
def generate_template(data):
    pred_template = []
    for patient in data["Patient"].unique():
        df = pd.DataFrame(columns=["PatientID", "Weeks", "Patient", "Class"])
        df["Weeks"] = np.arange(-12, 134)
        df["Patient"] = patient
        df["Class"] = int(data.loc[data["Patient"] == patient, "Class"].max())
        pred_template.append(df)
    pred_template = pd.concat(pred_template, ignore_index=True)
    pred_template["PatientID"] = le_id.transform(pred_template["Patient"])
    return pred_template


template_train_test = generate_template(test)
template_train_test.head()




,PatientID,Weeks,Patient,Class
0,5,-12,ID00014637202177757139317,1
1,5,-11,ID00014637202177757139317,1
2,5,-10,ID00014637202177757139317,1
3,5,-9,ID00014637202177757139317,1
4,5,-8,ID00014637202177757139317,1


In [8]:
def model_predict(model, trace, template):
    params = model["params"]
    sigma_by_class = model["sigma_by_class"]
    global_sigma = model["global_sigma"]

    pid = template["PatientID"].values.astype(int)
    w = template["Weeks"].values.astype(float)
    cls = template["Class"].values.astype(int)

    a = np.array([params.get(int(p), (np.nan, 0.0))[0] for p in pid], dtype=float)
    b = np.array([params.get(int(p), (np.nan, 0.0))[1] for p in pid], dtype=float)

    baseline_map = (
        pd.concat(
            [train[["PatientID", "FVC_base"]], test[["PatientID", "FVC_base"]]], axis=0
        )
        .drop_duplicates("PatientID")
        .set_index("PatientID")["FVC_base"]
        .to_dict()
    )

    missing = np.isnan(a)
    if missing.any():
        for i in np.where(missing)[0]:
            a[i] = float(baseline_map.get(int(pid[i]), 2000.0))
            b[i] = 0.0

    fvc_pred = a + b * w
    sigma = np.array(
        [sigma_by_class.get(int(c), global_sigma) for c in cls], dtype=float
    )

    df = pd.DataFrame(
        {
            "Patient": template["Patient"].values,
            "Weeks": template["Weeks"].values.astype(int),
            "FVC_pred": fvc_pred.astype(float),
            "sigma": sigma.astype(float),
        }
    )
    df["FVC_inf"] = df["FVC_pred"] - df["sigma"]
    df["FVC_sup"] = df["FVC_pred"] + df["sigma"]

    df = pd.merge(
        df, train[["Patient", "Weeks", "FVC"]], how="left", on=["Patient", "Weeks"]
    )
    df = df.rename(columns={"FVC": "FVC_true"})
    return df




In [9]:
import seaborn as sns
import matplotlib.pyplot as plt


def examine_predictions(data):
    n = (data["Patient"].nunique()) + 1
    f, axes = plt.subplots((n // 3) + 1, 3, figsize=(15, 5 * ((n // 3) + 1)))
    for i, patient in enumerate(data["Patient"].unique()):
        ax = axes[i // 3, i % 3]
        df = data[data["Patient"] == patient]
        x = df["Weeks"]
        ax.set_title(patient)
        ax.plot(x, df["FVC_true"], "o")
        ax.plot(x, df["FVC_pred"])
        sns.regplot(
            x=df["Weeks"], y=df["FVC_true"], ax=ax, ci=None, line_kws={"color": "red"}
        )
        ax.fill_between(x, df["FVC_inf"], df["FVC_sup"], alpha=0.5, color="#ffcd3c")
        ax.set_ylabel("FVC")
    axes[n // 3, n % 3].plot()




In [10]:
SIGMA_MULT = 2.0  # fallback default


def evaluate_predictions(
    df, use_only_last_3_measures=False, examine=False, sigma_mult=None
):
    if sigma_mult is None:
        sigma_mult = SIGMA_MULT

    if use_only_last_3_measures:
        y = df.dropna().groupby("Patient").tail(3).copy()
    else:
        y = df.dropna().copy()

    sigma_c = y["sigma"].values.astype(float) * float(sigma_mult)
    sigma_c[sigma_c < 70] = 70
    delta = (y["FVC_pred"] - y["FVC_true"]).abs().values.astype(float)
    delta[delta > 1000] = 1000
    lll = -np.sqrt(2) * delta / sigma_c - np.log(np.sqrt(2) * sigma_c)

    y["sigma_c"] = y["sigma"].astype(float) * float(sigma_mult)
    y.loc[y["sigma_c"] < 70, "sigma_c"] = 70
    y["delta_c"] = (y["FVC_pred"] - y["FVC_true"]).abs()
    y.loc[y["delta_c"] > 1000, "delta_c"] = 1000
    y["main_loss"] = y["delta_c"] / y["sigma_c"]

    if examine:
        plt.hist(y["main_loss"], bins=100)

    return float(np.mean(lll)) if len(lll) else np.nan


# Change (score-relevant, minimal): calibrate SIGMA_MULT using an OOF-style setup that mimics the test condition:
# per patient, fit on all-but-last-3 and evaluate only on the held-out last-3 rows.
# This keeps the exact same model core (per-patient linear fit + class residual sigma), only calibrating confidence scaling.
def calibrate_sigma_mult_oof_last3(train_df):
    # modestly denser grid than before to better hit a near-optimal sigma scale without changing model logic
    grid = [1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 3.2, 3.5, 4.0]

    # build OOF predictions for held-out last-3 rows per patient
    oof_frames = []
    for patient, g in train_df.groupby("Patient"):
        g = g.sort_values("Weeks").reset_index(drop=True)
        if len(g) <= 3:
            # not enough history to do holdout; skip to avoid degenerate fits
            continue

        g_train = g.iloc[:-3].copy()
        g_hold = g.iloc[-3:].copy()

        m, tr = model_fit(g_train, examine=False)

        # predict only for held-out weeks (not the full template) to speed up and to match scoring rows
        template_hold = g_hold[["PatientID", "Weeks", "Patient", "Class"]].copy()
        pred_hold = model_predict(m, tr, template_hold)

        # keep only the held-out actual rows (merge in model_predict already provides FVC_true)
        pred_hold = pred_hold.dropna(subset=["FVC_true"]).copy()
        oof_frames.append(pred_hold)

    if not oof_frames:
        return float(SIGMA_MULT), np.nan

    oof = pd.concat(oof_frames, ignore_index=True)

    best_m = None
    best_lll = -1e18
    for m in grid:
        lll = evaluate_predictions(oof, use_only_last_3_measures=False, sigma_mult=m)
        if np.isfinite(lll) and lll > best_lll:
            best_lll = lll
            best_m = m

    return float(best_m if best_m is not None else SIGMA_MULT), float(best_lll)




In [11]:
def evaluation_cycle(train_df, valid_df, examine_trace=True, examine_preds=True):
    print("Fit model ...")
    model, trace = model_fit(train_df, examine=examine_trace)
    print("")

    print("Examine true vs predictions for training data ...")
    template_train = generate_template(train_df)
    pred_train = model_predict(model, trace, template_train)
    if examine_preds:
        examine_predictions(pred_train)
    lll_train = evaluate_predictions(pred_train)

    pred_valid, lll_valid = None, None
    if valid_df is not None:
        print("Examine true vs predictions for validation data ...")
        template_valid = generate_template(valid_df)
        pred_valid = model_predict(model, trace, template_valid)
        if examine_preds:
            examine_predictions(pred_valid)
        lll_valid = evaluate_predictions(pred_valid)

    return pred_train, pred_valid, lll_train, lll_valid




In [12]:
for fold in range(0):
    examine_lll = True

    all_patients = train["Patient"].unique()
    validation_patients = np.random.choice(all_patients, size=20, replace=False)
    df_valid = train[train["Patient"].isin(validation_patients)]
    df_train = train[~train["Patient"].isin(validation_patients)]

    df_valid_first_readings = df_valid.groupby("Patient").head(1)
    df_train = pd.concat([df_train, df_valid_first_readings], axis=0, ignore_index=True)

    print(f"Fold: {fold}")
    pred_train, pred_valid, lll_train, lll_valid = evaluation_cycle(
        df_train, df_valid, examine_trace=True, examine_preds=True
    )

    print(f"Laplace Log Likelihoods for fold: {fold}")
    print(f"Training:     {lll_train:.4f}")
    print(f"Validation:   {lll_valid:.4f}")
    print("")

    evaluate_predictions(pred_train, use_only_last_3_measures=True, examine=examine_lll)
    evaluate_predictions(pred_valid, use_only_last_3_measures=True, examine=examine_lll)



In [13]:
print("Fit model ...")
model, trace = model_fit(train, examine=False)
print("")

# Change (score-relevant, minimal): choose SIGMA_MULT via OOF last-3 calibration (better matches leaderboard)
SIGMA_MULT, best_oof_lll = calibrate_sigma_mult_oof_last3(train)
print(f"Calibrated SIGMA_MULT={SIGMA_MULT:.3f} (OOF last-3 LLL={best_oof_lll:.6f})")

print("Make predictions for test data ...")
sub2 = sub.copy()
sub2[["Patient", "Weeks"]] = sub2["Patient_Week"].str.split("_", expand=True)
sub2["Weeks"] = sub2["Weeks"].astype(int)
sub2["PatientID"] = le_id.transform(sub2["Patient"])

class_map = test.groupby("Patient")["Class"].max().to_dict()
sub2["Class"] = sub2["Patient"].map(class_map).astype(int)

template_test = sub2[["PatientID", "Weeks", "Patient", "Class"]].copy()
pred_test = model_predict(model, trace, template_test)

final = sub[["Patient_Week"]].copy()
final[["Patient", "Weeks"]] = final["Patient_Week"].str.split("_", expand=True)
final["Weeks"] = final["Weeks"].astype(int)

final = final.merge(
    pred_test[["Patient", "Weeks", "FVC_pred", "sigma"]],
    on=["Patient", "Weeks"],
    how="left",
)

final = final.rename(columns={"FVC_pred": "FVC", "sigma": "Confidence"})[
    ["Patient_Week", "FVC", "Confidence"]
]

final["FVC"] = final["FVC"].fillna(2000.0).astype(float)
final["Confidence"] = final["Confidence"].fillna(200.0).astype(float)

final["Confidence"] = (final["Confidence"] * SIGMA_MULT).clip(lower=70.0)

final.to_csv("submission.csv", index=False)
print(final.shape)
print(final.head())
print("Wrote submission.csv")

Fit model ...

Calibrated SIGMA_MULT=4.000 (OOF last-3 LLL=-7.679599)
Make predictions for test data ...
(1908, 3)
                   Patient_Week     FVC  Confidence
0  ID00126637202218610655908_-3  2375.0  498.987066
1  ID00126637202218610655908_-2  2375.0  498.987066
2  ID00126637202218610655908_-1  2375.0  498.987066
3   ID00126637202218610655908_0  2375.0  498.987066
4   ID00126637202218610655908_1  2375.0  498.987066
Wrote submission.csv
